# DATA STAGING

## SETUP

In [22]:
!pip install psycopg2-binary sqlalchemy --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import os

BASE_DIR        = 'items'
RAW_DIR         = f'{BASE_DIR}/raw_batches'
TRANSFORMED_DIR = f'{BASE_DIR}/transformed'
LOG_DIR         = f'{BASE_DIR}/logs'

for folder in [BASE_DIR, RAW_DIR, TRANSFORMED_DIR, LOG_DIR]:
    os.makedirs(folder, exist_ok=True)
    print(f' {folder}')

 items
 items/raw_batches
 items/transformed
 items/logs


In [24]:
BASE_CSV = f'{BASE_DIR}/opensky_data.csv'


### A. Fungsi Scraper OpenSky API (1x Fetch)

In [25]:
import requests
import pandas as pd
from datetime import datetime
import os

OPENSKY_COLUMNS = [
    'icao24', 'callsign', 'origin_country', 'time_position', 'last_contact',
    'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity',
    'true_track', 'vertical_rate', 'sensors', 'geo_altitude',
    'squawk', 'spi', 'position_source'
]
OPENSKY_URL = 'https://opensky-network.org/api/states/all'

def fetch_once(save_dir: str = RAW_DIR) -> str:
    timestamp_str = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
    out_path = f'{save_dir}/opensky_{timestamp_str}.csv'
    try:
        print(f' Fetching OpenSky API [{timestamp_str} UTC]')
        resp = requests.get(OPENSKY_URL, timeout=30)
        resp.raise_for_status()
        data   = resp.json()
        states = data.get('states', [])
        if not states:
            print('API mengembalikan data kosong.')
            return None
        df = pd.DataFrame(states, columns=OPENSKY_COLUMNS)
        df['scraped_at'] = datetime.utcnow().isoformat()
        df.to_csv(out_path, index=False)
        print(f'Tersimpan: {out_path} | Baris: {len(df):,}')
        return out_path
    except requests.exceptions.RequestException as e:
        print(f'Gagal fetch: {e}')
        return None

print('Mencoba fetch 1x dari OpenSky API...')
result = fetch_once()
if result:
    print(f'\n Scraping berhasil: {result}')
else:
    print('\nGagal')

Mencoba fetch 1x dari OpenSky API...
 Fetching OpenSky API [20260524_102359 UTC]
Tersimpan: items/raw_batches/opensky_20260524_102359.csv | Baris: 6,940

 Scraping berhasil: items/raw_batches/opensky_20260524_102359.csv


### B. Simulasi Scraping Periodik 2 Tahun

In [26]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

N_BATCHES        = 4384         
INTERVAL_HOURS   = 6
START_DATE       = datetime(2023, 1, 1, 0, 0, 0)
ROWS_PER_BATCH   = 150           
                                  
ADD_NOISE        = True
RANDOM_SEED      = 42

np.random.seed(RANDOM_SEED)

print(f'Membaca base CSV: {BASE_CSV}')
df_base = pd.read_csv(BASE_CSV, low_memory=False)
if 'scraped_at' not in df_base.columns:
    df_base['scraped_at'] = START_DATE.isoformat()
print(f'   Base shape: {df_base.shape}')

def add_realistic_noise(df, batch_dt, n_rows):
    """Sample n_rows baris + tambahkan variasi realistis."""
    dfc = df.sample(
        n=min(n_rows, len(df)),
        random_state=int(batch_dt.timestamp()) % 99999,
        replace=False
    ).copy()

    batch_ts = batch_dt.timestamp()
    dfc['time_position'] = batch_ts + np.random.uniform(-60, 60, size=len(dfc))
    dfc['last_contact']  = batch_ts + np.random.uniform(0, 30, size=len(dfc))

    if ADD_NOISE:
        dfc['longitude']     = dfc['longitude'].astype(float)     + np.random.normal(0, 0.05, size=len(dfc))
        dfc['latitude']      = dfc['latitude'].astype(float)      + np.random.normal(0, 0.05, size=len(dfc))
        dfc['velocity']      = dfc['velocity'].astype(float)      + np.random.normal(0, 5,    size=len(dfc))
        dfc['baro_altitude'] = dfc['baro_altitude'].astype(float) + np.random.normal(0, 50,   size=len(dfc))

    dfc['scraped_at'] = batch_dt.isoformat()
    return dfc.reset_index(drop=True)


# Estimasi storage sebelum generate
est_rows   = N_BATCHES * ROWS_PER_BATCH
est_csv_mb = (est_rows * 200) / (1024**2)  
est_db_mb  = (est_rows * 120) / (1024**2)  

print(f'   Batch       : {N_BATCHES:,} (tiap {INTERVAL_HOURS} jam)')
print(f'   Baris/batch : {ROWS_PER_BATCH}')
print(f'   Total baris : {est_rows:,}')
print(f'   CSV di Drive: ~{est_csv_mb:.0f} MB')
print(f'   DB Supabase : ~{est_db_mb:.0f} MB  (free tier limit: 500 MB)')
print(f'   Rentang     : {START_DATE.date()} → {(START_DATE + timedelta(hours=INTERVAL_HOURS*N_BATCHES)).date()}\n')

generated = []

for i in range(N_BATCHES):
    batch_dt = START_DATE + timedelta(hours=INTERVAL_HOURS * i)
    ts_str   = batch_dt.strftime('%Y%m%d_%H%M%S')
    out_path = f'{RAW_DIR}/opensky_{ts_str}.csv'

    if os.path.exists(out_path):
        generated.append(out_path)
        continue

    df_batch = add_realistic_noise(df_base, batch_dt, ROWS_PER_BATCH)
    df_batch.to_csv(out_path, index=False)
    generated.append(out_path)

    if (i + 1) % 500 == 0 or i == N_BATCHES - 1:
        print(f'   [{i+1:>4}/{N_BATCHES}] {ts_str} | {len(df_batch)} baris/file')

print(f'\nSelesai! Total batch: {len(generated):,}')
print(f'Lokasi: {RAW_DIR}')

Membaca base CSV: items/opensky_data.csv
   Base shape: (6564, 18)
   Batch       : 4,384 (tiap 6 jam)
   Baris/batch : 150
   Total baris : 657,600
   CSV di Drive: ~125 MB
   DB Supabase : ~75 MB  (free tier limit: 500 MB)
   Rentang     : 2023-01-01 → 2026-01-01


Selesai! Total batch: 4,384
Lokasi: items/raw_batches


### C. Verifikasi Hasil Simulasi

In [27]:
import os, pandas as pd

batch_files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith('.csv')])
print(f'   Total file   : {len(batch_files):,}')
print(f'   File pertama : {batch_files[0]}')
print(f'   File terakhir: {batch_files[-1]}')

for label, fname in [('PERTAMA', batch_files[0]), ('TERAKHIR', batch_files[-1])]:
    df_check = pd.read_csv(f'{RAW_DIR}/{fname}')
    print(f'\n  [{label}] {fname}')
    print(f'   Baris      : {len(df_check):,}')
    print(f'   scraped_at : {df_check["scraped_at"].iloc[0]}')
    print(f'   Negara unik: {df_check["origin_country"].nunique()}')


   Total file   : 4,388
   File pertama : opensky_20230101_000000.csv
   File terakhir: opensky_20260524_102359.csv

  [PERTAMA] opensky_20230101_000000.csv
   Baris      : 150
   scraped_at : 2023-01-01T00:00:00
   Negara unik: 23

  [TERAKHIR] opensky_20260524_102359.csv
   Baris      : 6,940
   scraped_at : 2026-05-24T10:24:03.274739
   Negara unik: 106


## 1. Reader / Parser (Extract)

### A. Load & Parsing Semua Batch CSV

In [28]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

KEY_COLUMNS = [
    'icao24', 'callsign', 'origin_country',
    'time_position', 'longitude', 'latitude',
    'baro_altitude', 'on_ground', 'velocity',
    'vertical_rate', 'scraped_at'
]

def parse_batch_lean(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath, low_memory=False)

    available = [c for c in KEY_COLUMNS if c in df.columns]
    df = df[available].copy()

    df['time_position'] = pd.to_numeric(df['time_position'], errors='coerce')
    df['datetime_utc']  = pd.to_datetime(
        df['time_position'], unit='s', utc=True, errors='coerce'
    )

    if 'scraped_at' in df.columns:
        df['scraped_at'] = pd.to_datetime(
            df['scraped_at'], errors='coerce', infer_datetime_format=True
        )

    df['source_file'] = os.path.basename(filepath)
    return df


batch_files = sorted([
    os.path.join(RAW_DIR, f)
    for f in os.listdir(RAW_DIR) if f.endswith('.csv')
])

print(f'Total file batch: {len(batch_files):,}')

all_batches = []
errors      = []

for i, fpath in enumerate(batch_files):
    try:
        df_chunk = parse_batch_lean(fpath)
        all_batches.append(df_chunk)
    except Exception as e:
        errors.append((fpath, str(e)))

    if (i + 1) % 20 == 0 or i == len(batch_files) - 1:
        total_rows = sum(len(x) for x in all_batches)
        print(f'   [{i+1:>4}/{len(batch_files)}] Total baris terkumpul: {total_rows:,}')

df_raw = pd.concat(all_batches, ignore_index=True)

del all_batches

print(f'   Total baris  : {len(df_raw):,}')
print(f'   Total kolom  : {len(df_raw.columns)}')
print(f'   RAM DataFrame: {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB')
if errors:
    print(f'Error file: {len(errors)}')

Total file batch: 4,388
   [  20/4388] Total baris terkumpul: 3,000
   [  40/4388] Total baris terkumpul: 6,000
   [  60/4388] Total baris terkumpul: 9,000
   [  80/4388] Total baris terkumpul: 12,000
   [ 100/4388] Total baris terkumpul: 15,000
   [ 120/4388] Total baris terkumpul: 18,000
   [ 140/4388] Total baris terkumpul: 21,000
   [ 160/4388] Total baris terkumpul: 24,000
   [ 180/4388] Total baris terkumpul: 27,000
   [ 200/4388] Total baris terkumpul: 30,000
   [ 220/4388] Total baris terkumpul: 33,000
   [ 240/4388] Total baris terkumpul: 36,000
   [ 260/4388] Total baris terkumpul: 39,000
   [ 280/4388] Total baris terkumpul: 42,000
   [ 300/4388] Total baris terkumpul: 45,000
   [ 320/4388] Total baris terkumpul: 48,000
   [ 340/4388] Total baris terkumpul: 51,000
   [ 360/4388] Total baris terkumpul: 54,000
   [ 380/4388] Total baris terkumpul: 57,000
   [ 400/4388] Total baris terkumpul: 60,000
   [ 420/4388] Total baris terkumpul: 63,000
   [ 440/4388] Total baris terkump

### B. Parsing Timestamp (Unix → Datetime UTC)

In [29]:
df_raw['time_position'] = pd.to_numeric(df_raw['time_position'], errors='coerce')
df_raw['datetime_utc']  = pd.to_datetime(df_raw['time_position'], unit='s', utc=True, errors='coerce')
df_raw['scraped_at']    = pd.to_datetime(df_raw['scraped_at'], errors='coerce')

display(df_raw[['icao24','callsign','time_position','datetime_utc','scraped_at']].head(5))
print(f'\n   Rentang data:')
print(f'   Terlama : {df_raw["datetime_utc"].min()}')
print(f'   Terbaru : {df_raw["datetime_utc"].max()}')

,icao24,callsign,time_position,datetime_utc,scraped_at
0,aa55b5,SKMX765,1.672506e+09,2022-12-31 16:59:44.944814205+00:00,2023-01-01
1,a3bda6,DAL2614,1.672506e+09,2022-12-31 17:00:54.085716724+00:00,2023-01-01
2,4bb294,TKJ206,1.672506e+09,2022-12-31 17:00:27.839272976+00:00,2023-01-01
3,ab4372,N825AD,1.672506e+09,2022-12-31 17:00:11.839018106+00:00,2023-01-01
4,a05c45,N122MT,1.672506e+09,2022-12-31 16:59:18.722236872+00:00,2023-01-01



   Rentang data:
   Terlama : 2022-12-31 16:59:00.662654161+00:00
   Terbaru : 2026-05-24 10:24:00+00:00


## 2.Pre-processor / Transform

### A. Hapus Duplikat & Null Kritis

In [30]:
print(f'Shape sebelum cleaning: {df_raw.shape}')
df_clean = df_raw.copy()

# Hapus duplikat
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['icao24','time_position'])
print(f'   Duplikat dihapus   : {before - len(df_clean):,} baris')

# Hapus baris dengan kolom kritis kosong
CRITICAL_COLS = ['icao24','origin_country','longitude','latitude','on_ground']
before = len(df_clean)
df_clean = df_clean.dropna(subset=CRITICAL_COLS)
print(f'   Null kritis dihapus: {before - len(df_clean):,} baris')

# Isi null non-kritis
df_clean['callsign']      = df_clean['callsign'].str.strip().fillna('UNKNOWN')
df_clean['baro_altitude'] = df_clean['baro_altitude'].fillna(0.0)
df_clean['velocity']      = df_clean['velocity'].fillna(0.0)
df_clean['vertical_rate'] = df_clean['vertical_rate'].fillna(0.0)

print(f'\nShape setelah cleaning: {df_clean.shape}')

Shape sebelum cleaning: (691580, 13)
   Duplikat dihapus   : 62 baris
   Null kritis dihapus: 8,316 baris

Shape setelah cleaning: (683202, 13)


### B. Normalisasi Nama Negara & Ekstrak Kode Maskapai

In [31]:
import re

COUNTRY_ALIASES = {
    'Usa': 'United States', 'Us': 'United States',
    'Uk': 'United Kingdom', 'Great Britain': 'United Kingdom',
    'Russian Federation': 'Russia', 'Viet Nam': 'Vietnam',
    'Korea, Republic Of': 'South Korea',
    'Iran, Islamic Republic Of': 'Iran',
}

def normalize_country(name):
    if pd.isna(name): return 'Unknown'
    name = str(name).strip().title()
    return COUNTRY_ALIASES.get(name, name)

def normalize_callsign(cs):
    if pd.isna(cs): return 'UNKNOWN'
    cs = re.sub(r'\s+', '', str(cs)).upper()
    return cs if cs else 'UNKNOWN'

df_clean['origin_country'] = df_clean['origin_country'].apply(normalize_country)
df_clean['callsign']       = df_clean['callsign'].apply(normalize_callsign)
df_clean['airline_code']   = df_clean['callsign'].str[:3].where(
    df_clean['callsign'].str.match(r'^[A-Z]{3}'), other='UNK'
)

print(f'   Negara unik  : {df_clean["origin_country"].nunique():,}')
print(f'   Maskapai unik: {df_clean["airline_code"].nunique():,}')
display(df_clean[['callsign','airline_code','origin_country']].drop_duplicates().head(8))

   Negara unik  : 127
   Maskapai unik: 2,308


,callsign,airline_code,origin_country
0,SKMX765,SKM,United States
1,DAL2614,DAL,United States
2,TKJ206,TKJ,Turkey
3,N825AD,UNK,United States
4,N122MT,UNK,United States
5,LEADER5,LEA,United Kingdom
6,CKS217,CKS,United States
7,AAL2003,AAL,United States


### C. Time Granularity (Ekstrak Komponen Waktu)

In [32]:
df_clean['date']        = df_clean['datetime_utc'].dt.date
df_clean['year']        = df_clean['datetime_utc'].dt.year
df_clean['month']       = df_clean['datetime_utc'].dt.month
df_clean['month_name']  = df_clean['datetime_utc'].dt.strftime('%B')
df_clean['day']         = df_clean['datetime_utc'].dt.day
df_clean['hour']        = df_clean['datetime_utc'].dt.hour
df_clean['day_of_week'] = df_clean['datetime_utc'].dt.day_name()
df_clean['quarter']     = df_clean['datetime_utc'].dt.quarter
df_clean['is_weekend']  = df_clean['datetime_utc'].dt.dayofweek >= 5

def time_of_day(h):
    if   0 <= h <  6: return 'Night'
    elif 6 <= h < 12: return 'Morning'
    elif 12<= h < 18: return 'Afternoon'
    else:              return 'Evening'

df_clean['time_of_day'] = df_clean['hour'].apply(time_of_day)

display(df_clean[['datetime_utc','date','year','month','day','hour','day_of_week','quarter','time_of_day','is_weekend']].head(5))

,datetime_utc,date,year,month,day,hour,day_of_week,quarter,time_of_day,is_weekend
0,2022-12-31 16:59:44.944814205+00:00,2022-12-31,2022,12,31,16,Saturday,4,Afternoon,True
1,2022-12-31 17:00:54.085716724+00:00,2022-12-31,2022,12,31,17,Saturday,4,Afternoon,True
2,2022-12-31 17:00:27.839272976+00:00,2022-12-31,2022,12,31,17,Saturday,4,Afternoon,True
3,2022-12-31 17:00:11.839018106+00:00,2022-12-31,2022,12,31,17,Saturday,4,Afternoon,True
4,2022-12-31 16:59:18.722236872+00:00,2022-12-31,2022,12,31,16,Saturday,4,Afternoon,True


### D. Klasifikasi Fase Penerbangan

In [33]:
def classify_flight_phase(row):
    on_ground  = row['on_ground']
    altitude   = float(row['baro_altitude'])  if pd.notna(row['baro_altitude'])  else 0.0
    vert_rate  = float(row['vertical_rate'])  if pd.notna(row['vertical_rate'])  else 0.0

    if on_ground or altitude <= 50:                   return 'Ground'
    if vert_rate >  2.0 and altitude <  3000:          return 'Takeoff'
    if vert_rate >  2.0 and altitude >= 3000:          return 'Climb'
    if vert_rate < -2.0 and altitude >= 3000:          return 'Descent'
    if vert_rate < -2.0 and altitude <  3000:          return 'Landing'
    if altitude  >= 6000 and abs(vert_rate) <= 2.0:    return 'Cruise'
    return 'Low_Flight'

print('Mengklasifikasi fase penerbangan')
df_clean['flight_phase'] = df_clean.apply(classify_flight_phase, axis=1)

print('\n Distribusi fase penerbangan:')
phase_counts = df_clean['flight_phase'].value_counts()
for phase, count in phase_counts.items():
    print(f'   {phase:<12}: {count:>8,}  ({count/len(df_clean)*100:.1f}%)')


Mengklasifikasi fase penerbangan

 Distribusi fase penerbangan:
   Cruise      :  274,364  (40.2%)
   Low_Flight  :  106,903  (15.6%)
   Ground      :   82,893  (12.1%)
   Climb       :   65,172  (9.5%)
   Landing     :   62,752  (9.2%)
   Descent     :   60,862  (8.9%)
   Takeoff     :   30,256  (4.4%)


### E. Geo-Region Mapping (Lat/Lon → Region & Benua)

In [34]:
def map_region(lat, lon):
    try: lat, lon = float(lat), float(lon)
    except: return ('Unknown', 'Unknown')
    if  35 <= lat <= 72  and -10  <= lon <= 40:   return ('Europe',            'Europe')
    if  15 <= lat <= 72  and -170 <= lon <= -50:  return ('North America',     'Americas')
    if -60 <= lat <  15  and -82  <= lon <= -34:  return ('South America',     'Americas')
    if  20 <= lat <= 55  and  100 <= lon <= 145:  return ('East Asia',         'Asia')
    if -10 <= lat <= 20  and  95  <= lon <= 140:  return ('Southeast Asia',    'Asia')
    if   5 <= lat <= 35  and  60  <= lon <= 95:   return ('South Asia',        'Asia')
    if  15 <= lat <= 42  and  35  <= lon <= 63:   return ('Middle East',       'Asia')
    if -35 <= lat <= 37  and -20  <= lon <= 52:   return ('Africa',            'Africa')
    if -50 <= lat <= 0   and  110 <= lon <= 180:  return ('Oceania',           'Oceania')
    if lat > 55          and  lon > 40:           return ('Russia/Central Asia','Asia')
    return ('Other', 'Unknown')

print('Memetakan koordinat ke region')
regions = df_clean.apply(lambda r: map_region(r['latitude'], r['longitude']), axis=1)
df_clean['region']    = regions.apply(lambda x: x[0])
df_clean['continent'] = regions.apply(lambda x: x[1])

print('Distribusi region:')
for region, count in df_clean['region'].value_counts().items():
    print(f'   {region:<25}: {count:>8,}  ({count/len(df_clean)*100:.1f}%)')

Memetakan koordinat ke region
Distribusi region:
   North America            :  450,794  (66.0%)
   Europe                   :   57,024  (8.3%)
   East Asia                :   52,655  (7.7%)
   Oceania                  :   41,040  (6.0%)
   South Asia               :   23,807  (3.5%)
   Southeast Asia           :   22,625  (3.3%)
   South America            :   12,534  (1.8%)
   Africa                   :    8,454  (1.2%)
   Middle East              :    6,686  (1.0%)
   Other                    :    5,473  (0.8%)
   Russia/Central Asia      :    2,110  (0.3%)


In [35]:
import os
transform_path = f'{TRANSFORMED_DIR}/opensky_transformed.csv'
df_clean.to_csv(transform_path, index=False)
size_mb = os.path.getsize(transform_path) / (1024*1024)
print(f'   Path  : {transform_path}')
print(f'   Baris : {len(df_clean):,}')
print(f'   Kolom : {len(df_clean.columns)}')
print(f'   Size  : {size_mb:.2f} MB')
print(f'\nKolom tersedia:')
print([col for col in df_clean.columns])

   Path  : items/transformed/opensky_transformed.csv
   Baris : 683,202
   Kolom : 27
   Size  : 193.91 MB

Kolom tersedia:
['icao24', 'callsign', 'origin_country', 'time_position', 'longitude', 'latitude', 'baro_altitude', 'on_ground', 'velocity', 'vertical_rate', 'scraped_at', 'datetime_utc', 'source_file', 'airline_code', 'date', 'year', 'month', 'month_name', 'day', 'hour', 'day_of_week', 'quarter', 'is_weekend', 'time_of_day', 'flight_phase', 'region', 'continent']


## 3. DDL Star Schema di Supabase PostgreSQL

### A. Koneksi ke Supabase

In [36]:
from sqlalchemy import create_engine, text

DB_USER = "postgres.vhdsgtlkrwvvoquegsqo"
DB_PASSWORD = "Botolair5171"
DB_HOST = "aws-1-ap-northeast-1.pooler.supabase.com"
DB_PORT = "6543"
DB_NAME = "postgres"

# Connection URL
DB_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    f"?sslmode=require"
)

try:
    engine = create_engine(
        DB_URL,
        pool_pre_ping=True
    )

    with engine.connect() as conn:
        version = conn.execute(
            text("SELECT version();")
        ).fetchone()[0]

    print("✅ Koneksi Transaction Pooler berhasil!")
    print(version[:100])

except Exception as e:
    print("❌ Gagal koneksi:")
    print(e)

✅ Koneksi Transaction Pooler berhasil!
PostgreSQL 17.6 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


### B. DDL: Buat Tabel, Partisi, Index, & Extension

In [37]:
DDL_STATEMENTS = [
    # Extensions
    'CREATE EXTENSION IF NOT EXISTS pg_stat_statements',
    'CREATE EXTENSION IF NOT EXISTS btree_gist',

    # dim_aircraft
    '''CREATE TABLE IF NOT EXISTS dim_aircraft (
        aircraft_id  SERIAL PRIMARY KEY,
        icao24       VARCHAR(10) NOT NULL UNIQUE,
        callsign     VARCHAR(20),
        airline_code VARCHAR(5),
        created_at   TIMESTAMP DEFAULT NOW()
    )''',

    # dim_country
    '''CREATE TABLE IF NOT EXISTS dim_country (
        country_id   SERIAL PRIMARY KEY,
        country_name VARCHAR(100) NOT NULL UNIQUE,
        region       VARCHAR(50),
        continent    VARCHAR(50)
    )''',

    # dim_time
    '''CREATE TABLE IF NOT EXISTS dim_time (
        time_id      SERIAL PRIMARY KEY,
        datetime_utc TIMESTAMP WITH TIME ZONE NOT NULL UNIQUE,
        date         DATE,
        year         SMALLINT,
        quarter      SMALLINT,
        month        SMALLINT,
        month_name   VARCHAR(10),
        day          SMALLINT,
        hour         SMALLINT,
        day_of_week  VARCHAR(10),
        time_of_day  VARCHAR(10),
        is_weekend   BOOLEAN
    )''',

    # dim_location
    '''CREATE TABLE IF NOT EXISTS dim_location (
        location_id SERIAL PRIMARY KEY,
        longitude   NUMERIC(9,5),
        latitude    NUMERIC(9,5),
        region      VARCHAR(50),
        continent   VARCHAR(50),
        lon_zone    VARCHAR(10),
        lat_zone    VARCHAR(10),
        CONSTRAINT dim_location_lon_lat_unique UNIQUE (longitude, latitude)
    )''',

    # fact_flight_state — location_id sekarang punya REFERENCES
    '''CREATE TABLE IF NOT EXISTS fact_flight_state (
        fact_id        BIGSERIAL,
        aircraft_id    INT REFERENCES dim_aircraft(aircraft_id),
        country_id     INT REFERENCES dim_country(country_id),
        time_id        INT REFERENCES dim_time(time_id),
        location_id    INT REFERENCES dim_location(location_id),
        baro_altitude  NUMERIC(10,2),
        velocity       NUMERIC(8,2),
        vertical_rate  NUMERIC(8,2),
        on_ground      BOOLEAN,
        flight_phase   VARCHAR(15),
        scraped_at     TIMESTAMP WITH TIME ZONE,
        partition_date DATE NOT NULL
    ) PARTITION BY RANGE (partition_date)''',

    # Partisi per tahun
    '''CREATE TABLE IF NOT EXISTS fact_flight_state_2023
        PARTITION OF fact_flight_state
        FOR VALUES FROM (\'2023-01-01\') TO (\'2024-01-01\')''',
    '''CREATE TABLE IF NOT EXISTS fact_flight_state_2024
        PARTITION OF fact_flight_state
        FOR VALUES FROM (\'2024-01-01\') TO (\'2025-01-01\')''',
    '''CREATE TABLE IF NOT EXISTS fact_flight_state_2025
        PARTITION OF fact_flight_state
        FOR VALUES FROM (\'2025-01-01\') TO (\'2026-01-01\')''',
    '''CREATE TABLE IF NOT EXISTS fact_flight_state_2026
        PARTITION OF fact_flight_state
        FOR VALUES FROM (\'2026-01-01\') TO (\'2027-01-01\')''',

    # BRIN Index
    'CREATE INDEX IF NOT EXISTS idx_fact_brin_date    ON fact_flight_state USING BRIN (partition_date)',
    'CREATE INDEX IF NOT EXISTS idx_fact_brin_scraped ON fact_flight_state USING BRIN (scraped_at)',

    # BTree Index untuk semua FK
    'CREATE INDEX IF NOT EXISTS idx_fact_aircraft ON fact_flight_state (aircraft_id)',
    'CREATE INDEX IF NOT EXISTS idx_fact_country  ON fact_flight_state (country_id)',
    'CREATE INDEX IF NOT EXISTS idx_fact_time     ON fact_flight_state (time_id)',
    'CREATE INDEX IF NOT EXISTS idx_fact_location ON fact_flight_state (location_id)',
    'CREATE INDEX IF NOT EXISTS idx_fact_phase    ON fact_flight_state (flight_phase)',
]

print('Menjalankan DDL Star Schema...')
with engine.connect() as conn:
    for stmt in DDL_STATEMENTS:
        try:
            conn.execute(text(stmt))
        except Exception as e:
            print(f'   error: {str(e)[:90]}')
    conn.commit()

print('\nStar Schema berhasil dibuat')
print('   Tabel   : dim_aircraft, dim_country, dim_time, dim_location, fact_flight_state')
print('   Partisi : 2023, 2024, 2025, 2026')
print('   FK      : aircraft_id, country_id, time_id, location_id')
print('   Index   : BRIN + BTree (semua FK columns)')

Menjalankan DDL Star Schema...

Star Schema berhasil dibuat
   Tabel   : dim_aircraft, dim_country, dim_time, dim_location, fact_flight_state
   Partisi : 2023, 2024, 2025, 2026
   FK      : aircraft_id, country_id, time_id, location_id
   Index   : BRIN + BTree (semua FK columns)


## 4. Feeder / Load ke Supabase

### A. Populate Tabel Dimensi

In [38]:
import pandas as pd
from sqlalchemy import text

# dim_aircraft 
df_ac = df_clean[['icao24','callsign','airline_code']].drop_duplicates(subset=['icao24'])
df_ac.to_sql('_stg_aircraft', engine, if_exists='replace', index=False)
with engine.connect() as conn:
    conn.execute(text('''
        INSERT INTO dim_aircraft (icao24, callsign, airline_code)
        SELECT icao24, callsign, airline_code FROM _stg_aircraft
        ON CONFLICT (icao24) DO UPDATE
            SET callsign=EXCLUDED.callsign, airline_code=EXCLUDED.airline_code
    '''))
    conn.commit()
print(f'dim_aircraft : {len(df_ac):,} records')

# dim_country 
df_co = df_clean[['origin_country','region','continent']].drop_duplicates(subset=['origin_country'])
df_co = df_co.rename(columns={'origin_country':'country_name'})
df_co.to_sql('_stg_country', engine, if_exists='replace', index=False)
with engine.connect() as conn:
    conn.execute(text('''
        INSERT INTO dim_country (country_name, region, continent)
        SELECT country_name, region, continent FROM _stg_country
        ON CONFLICT (country_name) DO NOTHING
    '''))
    conn.commit()
print(f'dim_country  : {len(df_co):,} records')

# dim_time
df_tm = df_clean[['datetime_utc','date','year','quarter','month','month_name',
                   'day','hour','day_of_week','time_of_day','is_weekend']].drop_duplicates(subset=['datetime_utc'])
df_tm.to_sql('_stg_time', engine, if_exists='replace', index=False)
with engine.connect() as conn:
    conn.execute(text('''
        INSERT INTO dim_time (datetime_utc,date,year,quarter,month,month_name,day,hour,day_of_week,time_of_day,is_weekend)
        SELECT datetime_utc,date,year,quarter,month,month_name,day,hour,day_of_week,time_of_day,is_weekend
        FROM _stg_time ON CONFLICT (datetime_utc) DO NOTHING
    '''))
    conn.commit()
print(f'dim_time     : {len(df_tm):,} records')

# dim_location 
df_lo = df_clean[['longitude','latitude','region','continent']].copy()
df_lo['longitude'] = df_lo['longitude'].round(2)
df_lo['latitude']  = df_lo['latitude'].round(2)
df_lo['lon_zone'] = df_lo['longitude'].apply(
    lambda x: f"{'E' if x>=0 else 'W'}{abs(int(x//10)*10)}-{abs(int(x//10)*10+10)}"
)
df_lo['lat_zone'] = df_lo['latitude'].apply(
    lambda x: f"{'N' if x>=0 else 'S'}{abs(int(x//10)*10)}-{abs(int(x//10)*10+10)}"
)
df_lo = df_lo.drop_duplicates(subset=['longitude','latitude'])
df_lo.to_sql('_stg_location', engine, if_exists='replace', index=False)

with engine.connect() as conn:
    conn.execute(text('''
        DO $$
        BEGIN
            IF NOT EXISTS (
                SELECT 1 FROM pg_constraint
                WHERE conname = 'dim_location_lon_lat_unique'
            ) THEN
                ALTER TABLE dim_location
                ADD CONSTRAINT dim_location_lon_lat_unique
                UNIQUE (longitude, latitude);
            END IF;
        END $$;
    '''))
    conn.execute(text('''
        INSERT INTO dim_location (longitude, latitude, region, continent, lon_zone, lat_zone)
        SELECT longitude, latitude, region, continent, lon_zone, lat_zone
        FROM _stg_location
        ON CONFLICT (longitude, latitude) DO NOTHING
    '''))
    conn.commit()
print(f'dim_location : {len(df_lo):,} records')


dim_aircraft : 23,356 records
dim_country  : 127 records
dim_time     : 651,154 records
dim_location : 523,123 records


### B. Load Fact Table (Batch Insert)

In [39]:
import pandas as pd
from sqlalchemy import text

print('Menyusun fact table dengan lookup ID dimensi')

# Ambil mapping ID dari DB
aircraft_map = pd.read_sql('SELECT aircraft_id, icao24 FROM dim_aircraft', engine)\
                 .set_index('icao24')['aircraft_id'].to_dict()
country_map  = pd.read_sql('SELECT country_id, country_name FROM dim_country', engine)\
                 .set_index('country_name')['country_id'].to_dict()
time_df      = pd.read_sql('SELECT time_id, datetime_utc FROM dim_time', engine)
time_df['datetime_utc'] = pd.to_datetime(time_df['datetime_utc'], utc=True)
time_map     = time_df.set_index('datetime_utc')['time_id'].to_dict()

# ambil location_map dari DB 
loc_df = pd.read_sql('SELECT location_id, longitude, latitude FROM dim_location', engine)
loc_df['longitude'] = loc_df['longitude'].astype(float).round(2)
loc_df['latitude']  = loc_df['latitude'].astype(float).round(2)
loc_df['loc_key']   = loc_df['longitude'].astype(str) + '_' + loc_df['latitude'].astype(str)
location_map        = loc_df.set_index('loc_key')['location_id'].to_dict()
print(f'   location_map: {len(location_map):,} entri')

# Buat fact DataFrame
df_fact = df_clean.copy()
df_fact['aircraft_id']    = df_fact['icao24'].map(aircraft_map)
df_fact['country_id']     = df_fact['origin_country'].map(country_map)
df_fact['time_id']        = df_fact['datetime_utc'].map(time_map)
df_fact['partition_date'] = df_fact['date'].astype(str)

# map location_id
df_fact['lon_r'] = df_fact['longitude'].round(2)
df_fact['lat_r'] = df_fact['latitude'].round(2)
df_fact['loc_key'] = df_fact['lon_r'].astype(str) + '_' + df_fact['lat_r'].astype(str)
df_fact['location_id'] = df_fact['loc_key'].map(location_map)

null_loc = df_fact['location_id'].isna().sum()
print(f'   location_id null: {null_loc:,} baris (dari {len(df_fact):,})')

# Filter rentang partisi 2023–2026
df_fact['partition_date_dt'] = pd.to_datetime(df_fact['partition_date'])
before = len(df_fact)
df_fact = df_fact[
    (df_fact['partition_date_dt'] >= '2023-01-01') &
    (df_fact['partition_date_dt'] <  '2027-01-01')
]
filtered = before - len(df_fact)
if filtered > 0:
    print(f'{filtered:,} baris di luar rentang partisi dibuang')
df_fact = df_fact.drop(columns=['partition_date_dt','lon_r','lat_r','loc_key'])

fact_cols = ['aircraft_id','country_id','time_id','location_id',
             'baro_altitude','velocity','vertical_rate','on_ground',
             'flight_phase','scraped_at','partition_date']
df_fact = df_fact[fact_cols].dropna(subset=['aircraft_id','country_id','time_id'])

# Batch insert
BATCH_SIZE = 1000
total = len(df_fact)
print(f'\nMemasukkan {total:,} baris ke fact_flight_state...')
for start in range(0, total, BATCH_SIZE):
    chunk = df_fact.iloc[start:start+BATCH_SIZE]
    chunk.to_sql('fact_flight_state', engine, if_exists='append', index=False, method='multi')
    if (start//BATCH_SIZE+1) % 50 == 0 or start+BATCH_SIZE >= total:
        pct = min(start+BATCH_SIZE, total)/total*100
        print(f'   [{min(start+BATCH_SIZE,total):>7,}/{total:,}] {pct:.1f}%')

print('\nFact table berhasil diisi dengan location_id')

Menyusun fact table dengan lookup ID dimensi
   location_map: 523,123 entri
   location_id null: 9 baris (dari 683,202)
297 baris di luar rentang partisi dibuang

Memasukkan 34,231 baris ke fact_flight_state...
   [ 34,231/34,231] 100.0%

Fact table berhasil diisi dengan location_id


### C. Materialized Views (Pre-computed OLAP)

In [40]:
MV_STATEMENTS = [
    # MV 1: Penerbangan per negara per jam
    '''CREATE MATERIALIZED VIEW IF NOT EXISTS mv_flight_by_country_hour AS
    SELECT dc.country_name, dc.region, dc.continent,
           dt.year, dt.month, dt.day, dt.hour, dt.time_of_day,
           COUNT(*)                              AS total_flights,
           AVG(f.velocity)                       AS avg_velocity,
           AVG(f.baro_altitude)                  AS avg_altitude,
           SUM(CASE WHEN f.on_ground THEN 1 ELSE 0 END) AS on_ground_count
    FROM fact_flight_state f
    JOIN dim_country dc ON f.country_id = dc.country_id
    JOIN dim_time    dt ON f.time_id    = dt.time_id
    GROUP BY dc.country_name, dc.region, dc.continent,
             dt.year, dt.month, dt.day, dt.hour, dt.time_of_day
    WITH DATA''',

    # MV 2: Fase penerbangan harian
    '''CREATE MATERIALIZED VIEW IF NOT EXISTS mv_flight_phase_daily AS
    SELECT dt.date, dt.year, dt.month, f.flight_phase,
           COUNT(*)             AS total_count,
           AVG(f.baro_altitude) AS avg_altitude,
           AVG(f.velocity)      AS avg_velocity,
           AVG(f.vertical_rate) AS avg_vertical_rate
    FROM fact_flight_state f
    JOIN dim_time dt ON f.time_id = dt.time_id
    GROUP BY dt.date, dt.year, dt.month, f.flight_phase
    WITH DATA''',

    # MV 3: Top maskapai per region
    '''CREATE MATERIALIZED VIEW IF NOT EXISTS mv_airline_by_region AS
    SELECT da.airline_code, dc.region, dc.continent,
           dt.year, dt.month, COUNT(*) AS total_flights
    FROM fact_flight_state f
    JOIN dim_aircraft da ON f.aircraft_id = da.aircraft_id
    JOIN dim_country  dc ON f.country_id  = dc.country_id
    JOIN dim_time     dt ON f.time_id     = dt.time_id
    GROUP BY da.airline_code, dc.region, dc.continent, dt.year, dt.month
    WITH DATA''',

    # Index pada MV
    'CREATE INDEX IF NOT EXISTS idx_mv1_country ON mv_flight_by_country_hour (country_name)',
    'CREATE INDEX IF NOT EXISTS idx_mv1_hour    ON mv_flight_by_country_hour (hour)',
    'CREATE INDEX IF NOT EXISTS idx_mv2_date    ON mv_flight_phase_daily (date)',
    'CREATE INDEX IF NOT EXISTS idx_mv2_phase   ON mv_flight_phase_daily (flight_phase)',
    'CREATE INDEX IF NOT EXISTS idx_mv3_airline ON mv_airline_by_region (airline_code)',
]

print('Membuat Materialized Views')
with engine.connect() as conn:
    for stmt in MV_STATEMENTS:
        try:
            conn.execute(text(stmt))
        except Exception as e:
            print(f'   error:  {str(e)[:90]}')
    conn.commit()

print('\nMaterialized Views berhasil:')
print('   - mv_flight_by_country_hour')
print('   - mv_flight_phase_daily')
print('   - mv_airline_by_region')

Membuat Materialized Views

Materialized Views berhasil:
   - mv_flight_by_country_hour
   - mv_flight_phase_daily
   - mv_airline_by_region


### D. Performance Benchmark (Raw JOIN vs Materialized View)

In [41]:
import time, pandas as pd

def benchmark(label, sql, n=3):
    times = []
    with engine.connect() as conn:
        for _ in range(n):
            t0 = time.perf_counter()
            conn.execute(text(sql)).fetchall()
            times.append((time.perf_counter()-t0)*1000)
    avg = sum(times)/len(times)
    print(f'   [{label:<35}] {avg:>8.2f} ms  (n={n} runs)')
    return avg

Q_RAW = '''
    SELECT dc.country_name, dt.hour, COUNT(*) AS total
    FROM fact_flight_state f
    JOIN dim_country dc ON f.country_id = dc.country_id
    JOIN dim_time    dt ON f.time_id    = dt.time_id
    GROUP BY dc.country_name, dt.hour ORDER BY total DESC LIMIT 20
'''
Q_MV = '''
    SELECT country_name, hour, SUM(total_flights) AS total
    FROM mv_flight_by_country_hour
    GROUP BY country_name, hour ORDER BY total DESC LIMIT 20
'''

print('Performance Benchmark\n')
t_raw = benchmark('Raw JOIN (fact + dims)', Q_RAW)
t_mv  = benchmark('Materialized View',      Q_MV)
speedup = t_raw/t_mv if t_mv > 0 else 0
print(f'\n Speedup: {speedup:.1f}x lebih cepat dengan Materialized View')

df_bench = pd.DataFrame({'Metode':['Raw JOIN','Materialized View'],
                          'Waktu_ms':[round(t_raw,2),round(t_mv,2)],
                          'Speedup':[1.0, round(speedup,2)]})
bench_path = f'{LOG_DIR}/benchmark_result.csv'
df_bench.to_csv(bench_path, index=False)
print(f'\nHasil disimpan: {bench_path}')
display(df_bench)

Performance Benchmark

   [Raw JOIN (fact + dims)             ]   289.12 ms  (n=3 runs)
   [Materialized View                  ]   144.17 ms  (n=3 runs)

 Speedup: 2.0x lebih cepat dengan Materialized View

Hasil disimpan: items/logs/benchmark_result.csv


,Metode,Waktu_ms,Speedup
0,Raw JOIN,289.12,1.00
1,Materialized View,144.17,2.01


### E. Verifikasi Akhir: Cek Semua Tabel di Supabase

In [42]:
import pandas as pd

tables = ['dim_aircraft','dim_country','dim_time','dim_location',
          'fact_flight_state','mv_flight_by_country_hour',
          'mv_flight_phase_daily','mv_airline_by_region']

print('Ringkasan Database SkyWarehouse\n')
print(f'{"Tabel / View":<35} {"Jumlah Record":>15}')
print('─'*52)
with engine.connect() as conn:
    for tbl in tables:
        try:
            n = conn.execute(text(f'SELECT COUNT(*) FROM {tbl}')).fetchone()[0]
            print(f'{tbl:<35} {n:>15,}')
        except:
            print(f'{tbl:<35} {"ERROR":>15}')

print('\n' + '═'*52)
print('🎉 Phase 1 — Data Staging SELESAI!')
print('   Google Drive: SkyWarehouse/')
print('   ├── raw_batches/      (CSV per batch)')
print('   ├── transformed/      (opensky_transformed.csv)')
print('   └── logs/             (benchmark_result.csv)')
print('\n   Supabase: Star Schema siap untuk Phase 2 (Atoti DataMart)')

Ringkasan Database SkyWarehouse

Tabel / View                          Jumlah Record
────────────────────────────────────────────────────
dim_aircraft                                 23,356
dim_country                                     127
dim_time                                    651,154
dim_location                                523,123
fact_flight_state                            34,231
mv_flight_by_country_hour                       852
mv_flight_phase_daily                           604
mv_airline_by_region                          2,212

════════════════════════════════════════════════════
🎉 Phase 1 — Data Staging SELESAI!
   Google Drive: SkyWarehouse/
   ├── raw_batches/      (CSV per batch)
   ├── transformed/      (opensky_transformed.csv)
   └── logs/             (benchmark_result.csv)

   Supabase: Star Schema siap untuk Phase 2 (Atoti DataMart)
